# 01 Build the detection dataset

**Compute bucket: run on Colab or Kaggle (CPU is fine, about 20 minutes, needs ~5 GB of Drive).**

Builds a YOLO-format dataset for ~18 campus classes from public sources, plus any photos you took yourself, and writes
`reunite_dataset.zip` to Drive. Notebook `02` trains on it.

Where the classes come from:

| Class | Source |
|---|---|
| laptop, phone, tablet, headphones, backpack, bottle, book, spectacles, watch, umbrella, calculator | Open Images V7 (boxes, CC BY 2.0 images) |
| earbuds, charger, power_bank, wallet, keys, notebook | Open Images has no clean class. Use LVIS/Objects365 names below if your FiftyOne version has them, and top up with your own photos |
| id_card | no public source. Photograph 50+ real or dummy cards (never real people's cards on shared drives) |

**Your own photos** go in `MyDrive/reunite/own/<class>/*.jpg` with a YOLO `.txt` label next to each (label with CVAT,
Roboflow, or `labelImg`). Even 30 per weak class helps a lot.

In [ ]:
!pip -q install fiftyone ultralytics pyyaml pillow

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os, random, shutil, json, zipfile
from pathlib import Path
import yaml

CLASSES = [
    "id_card", "laptop", "phone", "tablet", "earbuds", "headphones", "charger", "power_bank", "wallet",
    "keys", "backpack", "bottle", "book", "notebook", "spectacles", "watch", "umbrella", "calculator",
]  # same order and names as backend/app/ml/text/lexicons/categories.yaml

WORK = Path("/content/work"); WORK.mkdir(exist_ok=True)
OUT = WORK / "dataset"
OWN = Path("/content/drive/MyDrive/reunite/own")
DEST = Path("/content/drive/MyDrive/reunite"); DEST.mkdir(parents=True, exist_ok=True)
PER_CLASS = 400          # Open Images images per class before de-duplication
VAL_SHARE = 0.15
random.seed(0)

In [ ]:
# campus class -> label names in the source datasets
OPEN_IMAGES = {
    "laptop": ["Laptop"], "phone": ["Mobile phone"], "tablet": ["Tablet computer"], "headphones": ["Headphones"],
    "backpack": ["Backpack"], "bottle": ["Bottle"], "book": ["Book"], "spectacles": ["Glasses"],
    "watch": ["Watch"], "umbrella": ["Umbrella"], "calculator": ["Calculator"],
}
LVIS = {  # only used if your FiftyOne zoo has 'lvis'; skipped quietly otherwise
    "earbuds": ["earphone"], "charger": ["charger"], "power_bank": ["battery"], "wallet": ["wallet"],
    "keys": ["key"], "notebook": ["notebook"],
}

In [ ]:
import fiftyone as fo
import fiftyone.zoo as foz

def load(source, mapping, split):
    names = sorted({n for v in mapping.values() for n in v})
    try:
        ds = foz.load_zoo_dataset(source, split=split, label_types=["detections"], classes=names,
                                  max_samples=PER_CLASS * max(len(mapping), 1), dataset_name=f"reunite_{source}_{split}", drop_existing_dataset=True)
    except Exception as e:                       # noqa: BLE001
        print("skipping", source, "->", e); return None
    back = {n: c for c, ns in mapping.items() for n in ns}
    for sample in ds:
        keep = []
        for det in sample.ground_truth.detections:
            if det.label not in back or getattr(det, "IsGroupOf", False):   # skip other classes and "a crowd of X" boxes
                continue
            det.label = back[det.label]
            keep.append(det)
        sample.ground_truth.detections = keep
        sample.save()
    return ds.match(fo.ViewField("ground_truth.detections").length() > 0)

parts = [p for p in (load("open-images-v7", OPEN_IMAGES, "train"), load("open-images-v7", OPEN_IMAGES, "validation"), load("lvis", LVIS, "train")) if p is not None]
print([len(p) for p in parts])

In [ ]:
# write YOLO layout: images/{train,val}, labels/{train,val}
for split in ("train", "val"):
    (OUT / "images" / split).mkdir(parents=True, exist_ok=True); (OUT / "labels" / split).mkdir(parents=True, exist_ok=True)

def write_sample(img_path, dets, split, tag):
    stem = f"{tag}_{Path(img_path).stem}"
    shutil.copy(img_path, OUT / "images" / split / f"{stem}.jpg")
    lines = []
    for label, (x, y, w, h) in dets:
        if label in CLASSES:
            lines.append(f"{CLASSES.index(label)} {x + w / 2:.6f} {y + h / 2:.6f} {w:.6f} {h:.6f}")
    (OUT / "labels" / split / f"{stem}.txt").write_text("\n".join(lines))

n = 0
for i, view in enumerate(parts):
    for s in view.iter_samples(progress=True):
        split = "val" if random.random() < VAL_SHARE else "train"
        write_sample(s.filepath, [(d.label, d.bounding_box) for d in s.ground_truth.detections], split, f"src{i}"); n += 1
print("public images:", n)

# your own photos: own/<class>/<name>.jpg + own/<class>/<name>.txt (YOLO format, class index per CLASSES)
own = 0
for cls_dir in OWN.glob("*"):
    for img in cls_dir.glob("*.jpg"):
        txt = img.with_suffix(".txt")
        if not txt.exists():
            continue
        split = "val" if random.random() < VAL_SHARE else "train"
        stem = f"own_{cls_dir.name}_{img.stem}"
        shutil.copy(img, OUT / "images" / split / f"{stem}.jpg"); shutil.copy(txt, OUT / "labels" / split / f"{stem}.txt"); own += 1
print("your photos:", own)

In [ ]:
# how balanced is it? weak classes need your own photos before you train
from collections import Counter
counts = Counter()
for f in (OUT / "labels" / "train").glob("*.txt"):
    for line in f.read_text().splitlines():
        counts[CLASSES[int(line.split()[0])]] += 1
for c in CLASSES:
    flag = "" if counts[c] >= 150 else "   <-- add photos"
    print(f"{c:12s} {counts[c]:5d}{flag}")

In [ ]:
(OUT / "dataset.yaml").write_text(yaml.safe_dump({"path": str(OUT), "train": "images/train", "val": "images/val", "names": {i: c for i, c in enumerate(CLASSES)}}))
zip_path = DEST / "reunite_dataset.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_STORED) as z:
    for p in OUT.rglob("*"):
        if p.is_file():
            z.write(p, p.relative_to(OUT))
json.dump({"classes": CLASSES, "counts": dict(counts), "sources": ["Open Images V7 (CC BY 2.0 images, CC BY 4.0 boxes)", "LVIS (if available)", "own photos"]}, open(DEST / "dataset_card.json", "w"), indent=2)
print("wrote", zip_path, zip_path.stat().st_size // 2**20, "MB")